In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from typing import TypedDict

import numpy as np
import numpy.typing as npt
import polars as pl
import torch
from tabulate import tabulate

from fart.model.nbeats_config import NBeatsConfig
from fart.model.nbeats_dataset import build_return_windows
from fart.model.nbeats_persistence import load_model
from fart.model.train_model import prepare_training_data, train
from fart.utils import get_latest_model_filepath, get_project_root

In [ ]:
assets_dir: Path = get_project_root() / "assets"
artifacts_dir: Path = get_project_root() / "artifacts"
market: str = "BTC-EUR"
interval: str = "1d"

N_RUNS_PER_BETA = 30
# beta=0.0 reproduces plain Gaussian NLL exactly (see
# tests/model/test_nbeats_loss.py::test_beta_nll_loss_at_beta_zero_matches_gaussian_nll_loss),
# so it stands in as the "Gaussian NLL baseline" arm within the same training path.
BETAS = [0.0, 0.5, 1.0]

In [ ]:
# Test windows are identical across every run in this sweep (lookback is
# held constant), so they're derived once here instead of after each of the
# training runs below.
_, _, y_train, y_test = prepare_training_data(
    data_dir=assets_dir, market=market, interval=interval, months=None
)
n_train: int = y_train.shape[0]
close_prices: pl.Series = pl.concat([y_train, y_test])
lookback: int = NBeatsConfig().lookback
X_all, y_all = build_return_windows(close_prices, lookback)
n_train_windows: int = max(0, n_train - lookback - 1)
X_test_windows: torch.Tensor = X_all[n_train_windows:]
y_test_windows: torch.Tensor = y_all[n_train_windows:]

In [ ]:
class Metrics(TypedDict):
    pearson: float
    spearman: float


def rank(a: npt.NDArray[np.float64]) -> npt.NDArray[np.intp]:
    return np.argsort(np.argsort(a))


def evaluate(model_path: Path) -> Metrics:
    model, _config = load_model(model_path)
    model.eval()
    with torch.no_grad():
        mu, log_sigma = model(X_test_windows).unbind(-1)
    confidence: npt.NDArray[np.float64] = (1 / (1 + log_sigma.exp())).numpy()
    error: npt.NDArray[np.float64] = (y_test_windows - mu).abs().numpy()
    return {
        "pearson": float(np.corrcoef(confidence, error)[0, 1]),
        "spearman": float(np.corrcoef(rank(confidence), rank(error))[0, 1]),
    }

In [ ]:
class RunResult(Metrics):
    beta: float
    run: int


results: list[RunResult] = []
for beta in BETAS:
    for run_idx in range(N_RUNS_PER_BETA):
        config = NBeatsConfig(beta_nll=beta)
        train(
            data_dir=assets_dir,
            market=market,
            interval=interval,
            artifacts_dir=artifacts_dir,
            months=None,
            config=config,
        )
        model_path = get_latest_model_filepath(artifacts_dir, market, interval)
        metrics = evaluate(model_path)
        results.append(
            RunResult(
                beta=beta,
                run=run_idx,
                pearson=metrics["pearson"],
                spearman=metrics["spearman"],
            )
        )

In [ ]:
rows: list[list[float | int | str]] = [
    [r["beta"], r["run"], f"{r['pearson']:.4f}", f"{r['spearman']:.4f}"]
    for r in results
]
print(tabulate(rows, headers=["beta", "run", "pearson", "spearman"]))

In [ ]:
# (mean, std, sem, n) per beta.
SummaryStats = tuple[float, float, float, int]

summary: dict[float, SummaryStats] = {}
for beta in BETAS:
    group: npt.NDArray[np.float64] = np.array(
        [r["pearson"] for r in results if r["beta"] == beta]
    )
    n: int = len(group)
    mean: float = float(group.mean())
    std: float = float(group.std(ddof=1))
    sem: float = std / n**0.5
    summary[beta] = (mean, std, sem, n)

summary_rows: list[list[float | int | str]] = [
    [beta, n, f"{mean:.4f}", f"{std:.4f}", f"{sem:.4f}"]
    for beta, (mean, std, sem, n) in summary.items()
]
print(tabulate(summary_rows, headers=["beta", "n", "mean_r", "std_r", "sem_r"]))

In [ ]:
# Welch's t-statistic between each pair of arms -- |t| below ~2 at n=30
# per arm means the difference in means is not distinguishable from noise.
betas_list: list[float] = list(summary.keys())
t_rows: list[list[str | float]] = []
for i in range(len(betas_list)):
    for j in range(i + 1, len(betas_list)):
        b1, b2 = betas_list[i], betas_list[j]
        m1, s1, _, n1 = summary[b1]
        m2, s2, _, n2 = summary[b2]
        t: float = (m1 - m2) / (s1**2 / n1 + s2**2 / n2) ** 0.5
        t_rows.append([f"{b1} vs {b2}", f"{t:.3f}", f"{m1 - m2:+.4f}"])
print(tabulate(t_rows, headers=["comparison", "welch_t", "mean_diff"]))